# Guards and recompiles
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week2/Guards_And_Recompiles_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 2, Module 2: Graph capture in real frameworks**

When TorchDynamo captures a function, it runs the Python control flow once and records a graph specialized to what it saw: the tensor shapes, dtypes and devices, and the values of Python flags. To make that graph safe to reuse, Dynamo installs **guards**, checks that must still hold on the next call. If every guard passes, the compiled graph runs. If one fails, Dynamo compiles a new graph for the new case.

This is the price of commitment from the lecture: the more a compiler fixes ahead of time, the more often a change in the input forces new work. In this notebook you trigger guard failures on purpose, read the recompile log that explains each one, and see what happens when shapes are not allowed to become dynamic. You finish with the shape-bucket example from the lecture.

**What you will do**
1. Count compiles with a small custom backend.
2. Change the shape, then the dtype, and read the recompile log.
3. Change a Python flag the function reads, and see the guard that caught it.
4. Turn off dynamic shapes and watch every new shape recompile.
5. Compute the padding cost of fixed shape buckets.

**Before you start:** any runtime works. A GPU is not required. The notebook takes under a minute.

## 1. A backend that counts compiles
`torch.compile` hands each captured graph to a backend. The backend below does no optimization. It prints the input shapes of each graph it receives and returns the graph unchanged, so every print is one compile.

`torch._logging.set_logs(recompiles=True)` makes Dynamo print the guard that failed each time it recompiles. That message is the most direct answer to the question "why did this recompile?"

Each log line starts with a compile ID in square brackets, such as `[0/1]`. The first number is the frame (which function), the second is the compile count for that frame. `[0/1]` is the second compile of the first function. tlparse reports use the same IDs, which Week 5 uses to find a recompile storm.

In [ ]:
import torch
import torch._dynamo

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

compiles = []


def counting_backend(gm, example_inputs):
    shapes = [tuple(t.shape) for t in example_inputs if isinstance(t, torch.Tensor)]
    compiles.append(shapes)
    print(f"    compile #{len(compiles)}: graph inputs {shapes}", flush=True)
    return gm.forward          # run the captured graph as it is


def call(fn, label, *args):
    before = len(compiles)
    fn(*args)
    if len(compiles) == before:
        what = "reused a compiled graph"
    else:
        what = "compiled" if before == 0 else "compiled again"
    print(f"{label:38} -> {what}", flush=True)


torch._logging.set_logs(recompiles=True)

## 2. Shapes and dtypes
The function is a small layer: multiply, add, ReLU, and a sum over the last dimension. The cell calls it with five inputs:

| Call | Input | What changes |
|---|---|---|
| 1 | float32, 8 by 128 | first call |
| 2 | float32, 8 by 128 | nothing |
| 3 | float32, 8 by 129 | the size of dimension 1 |
| 4 | float32, 8 by 130 | the size of dimension 1 again |
| 5 | float16, 8 by 130 | the dtype |

In [ ]:
def layer(x):
    return torch.relu(x * 2 + 1).sum(-1)


torch._dynamo.reset()
compiles.clear()
f = torch.compile(layer, backend=counting_backend)

call(f, "1. float32 (8, 128)", torch.randn(8, 128, device=device))
call(f, "2. float32 (8, 128) again", torch.randn(8, 128, device=device))
call(f, "3. float32 (8, 129)", torch.randn(8, 129, device=device))
call(f, "4. float32 (8, 130)", torch.randn(8, 130, device=device))
call(f, "5. float16 (8, 130)", torch.randn(8, 130, device=device, dtype=torch.float16))

**Reading the output**
- Call 2 passes every guard and reuses the graph. This is the fast path.
- Call 3 fails a size guard. The log names the tensor, the dimension, the expected size and the actual size.
- The second compile shows a symbol such as `s0` instead of a number in the input shape. After one size mismatch, PyTorch marks that dimension as **dynamic** and compiles a graph that works for any size. This is called automatic dynamic shapes.
- Call 4 reuses the dynamic graph, so a third size does not compile again.
- Call 5 fails the dtype guard. A dtype is never made dynamic, so every new dtype is a new graph.

## 3. A Python flag is a guard too
Dynamo runs ordinary Python `if` statements while it traces. The graph contains only the branch that was taken, and a guard records the value of the flag. If the flag changes, the guard fails.

In [ ]:
class Config:
    def __init__(self):
        self.use_bias = True


cfg = Config()


def projection(x, w, b):
    y = x @ w
    if cfg.use_bias:           # evaluated during tracing, not stored in the graph
        y = y + b
    return y


torch._dynamo.reset()
compiles.clear()
g = torch.compile(projection, backend=counting_backend)
x = torch.randn(4, 64, device=device)
w = torch.randn(64, 64, device=device)
b = torch.randn(64, device=device)

call(g, "use_bias = True", x, w, b)
call(g, "use_bias = True again", x, w, b)
cfg.use_bias = False
call(g, "use_bias = False", x, w, b)
cfg.use_bias = True
call(g, "use_bias = True, back again", x, w, b)

**Reading the output**
- The first graph contains the add. The second graph, compiled after the flag changed, does not.
- The recompile log names the guard on `cfg.use_bias`.
- Switching back to `True` reuses the first graph. Dynamo keeps a cache of compiled graphs per function and picks the one whose guards pass.

## 4. Turning dynamic shapes off
Accelerators that plan memory and schedules for fixed shapes compile with dynamic shapes off. `torch.compile(..., dynamic=False)` does the same here: every new size is a new graph.

Dynamo stops after a fixed number of recompiles for one function. Past that limit it prints a warning and runs the function in eager mode, without compiling. The cell prints the limit, then calls the layer with ten different sizes.

In [ ]:
limit = getattr(torch._dynamo.config, "recompile_limit", None) or getattr(torch._dynamo.config, "cache_size_limit", None)
print("Recompile limit per function:", limit)

torch._dynamo.reset()
compiles.clear()
h = torch.compile(layer, backend=counting_backend, dynamic=False)
for n in range(128, 138):
    call(h, f"dynamic=False, size {n}", torch.randn(8, n, device=device))
print(f"\nCompiles: {len(compiles)} for 10 different sizes")

**Reading the output**
- Each new size compiles a new graph until the recompile limit is reached.
- After that, Dynamo prints a warning that it hit the recompile limit, and the remaining calls run in eager mode. Nothing fails, but the function is no longer compiled. In a long run this warning is easy to miss.
- Week 5 and Week 6 return to this pattern as a **recompile storm**: a decode loop whose mask grows by one column every step.

## 5. Shape buckets
A system with fixed shapes avoids recompiling by padding each input up to one of a few compiled sizes, called **buckets**. The lecture example compiles buckets for 128 and 256 tokens. The cell computes the padding for three requests, then for a finer set of buckets.

In [ ]:
def pad_to_bucket(n, buckets):
    for b in sorted(buckets):
        if n <= b:
            return b
    raise ValueError(f"{n} tokens is larger than every bucket")


for buckets in ([128, 256], [64, 128, 192, 256]):
    print(f"buckets {buckets}")
    for n in (100, 250, 129):
        b = pad_to_bucket(n, buckets)
        print(f"  request {n:3d} tokens -> bucket {b:3d}, padding {b - n:3d} tokens ({(b - n) / b:5.1%} of the work)")

**Reading the output**
- With buckets of 128 and 256, a 129-token request pads to 256: 127 tokens, about half the work, is padding.
- Finer buckets cut the padding, but each bucket is another graph to compile, store and keep in memory.
- Choose the boundaries from the real distribution of request lengths. Week 6 Module 5 measures this trade in a serving loop.

## Try this
1. In section 2, call `f` with a tensor on the CPU after it was compiled on the GPU, if you have a GPU. Which guard fails?
2. In section 2, mark the dimension dynamic before the first call with `torch._dynamo.mark_dynamic(x, 1)`. How many compiles do calls 1 to 4 need now?
3. Turn on the full guard listing with `torch._logging.set_logs(guards=True)` and run section 3 again. Find the guard on `cfg.use_bias` and the `TENSOR_MATCH` guards on `x`, `w` and `b`.

## Summary
- A compiled graph is specialized to what Dynamo saw. Guards check that those assumptions still hold on every call.
- A shape change recompiles once, then the dimension becomes dynamic. A dtype change or a Python flag change recompiles every time.
- With `dynamic=False`, every new shape compiles until the recompile limit, then the function silently runs in eager mode.
- Fixed-shape systems pad to buckets instead. Fewer buckets mean more padding, more buckets mean more compiles.

Lecture: Week 2, Module 1, slides 20 and 21 (the price of commitment, shape buckets), and Module 2, slide 29 (guards).